# Fairness and Diagnostic Delay in Healthcare

Access to healthcare is not equally distributed across populations.

Patients may experience different waiting times before receiving a diagnosis
because of factors such as healthcare availability, geographic location,
economic resources, or administrative barriers.

These delays can have important consequences for health outcomes.

Machine learning systems developed using healthcare data may inherit
patterns created by unequal access to diagnosis and treatment.

In this notebook, we explore whether predictive performance differs across
groups that may be exposed to different risks of delayed diagnosis.

Rather than treating fairness as a purely technical problem,
we consider how social and institutional factors shape the data used
to train machine learning models.

# Healthcare Access and Stage at Cancer Diagnosis

Cancer outcomes are influenced not only by biological factors but also by
access to healthcare systems.

Patients facing economic, geographic, or administrative barriers may receive
a diagnosis at a more advanced stage of disease.

As machine learning systems are increasingly used in healthcare, it is
important to understand whether predictive models perform differently across
groups that experience unequal access to care.

In this notebook, we examine fairness in a cancer prediction task and explore
how disparities related to healthcare access may be reflected in model
performance.

Rather than treating fairness as a purely technical issue, we consider the
broader sociotechnical processes through which healthcare data are generated.

## Why healthcare access matters

Cancer is often more treatable when detected early.

However, opportunities for early detection are not equally distributed.
Access to screening programs, primary care providers, transportation,
health insurance, and specialist services can affect when a diagnosis is
made.

As a result, healthcare datasets may encode patterns that reflect
structural inequalities rather than purely medical differences.

Fairness tools can help practitioners identify whether predictive systems
behave differently across groups and support more informed decision-making.

# Fairness in Delayed Medical Care Due to Cost

Access to healthcare is not equally distributed across populations.

Even in systems where healthcare services are formally available,
individuals may experience financial barriers that lead to delayed care.

These delays are not random: they are shaped by income, insurance status,
and broader socioeconomic conditions.

In this notebook, we use data from the Medical Expenditure Panel Survey (MEPS)
to study delayed medical care due to cost.

We train a machine learning model to predict whether an individual reports
delaying medical care because of cost, and evaluate whether model performance
differs across income groups.

Rather than treating fairness as a purely technical constraint,
we analyze how socioeconomic inequality is reflected in model behavior.

# Fairness in Delayed Medical Care Due to Cost

Access to healthcare is not equally distributed across populations.

Even in systems where healthcare services are formally available,
individuals may experience financial barriers that lead to delayed care.

These delays are not random: they are shaped by income, insurance status,
and broader socioeconomic conditions.

In this notebook, we use data from the Medical Expenditure Panel Survey (MEPS)
to study delayed medical care due to cost.

We train a machine learning model to predict whether an individual reports
delaying medical care because of cost, and evaluate whether model performance
differs across income groups.

Rather than treating fairness as a purely technical constraint,
we analyze how socioeconomic inequality is reflected in model behavior.

# Mitigating Algorithmic Disparities in Healthcare Delays using IPUMS MEPS Data

## Introduction
Healthcare delay due to cost barrier is a critical socioeconomic challenge that disproportionately affects low-income populations. When medical diagnoses or treatments are postponed because of financial worries, it constitutes a form of systemic economic violence that exacerbates health inequities.

In this notebook, we demonstrate how to analyze and mitigate algorithmic unfairness in predictive models deployed in healthcare management. We use the **IPUMS Medical Expenditure Panel Survey (MEPS)** dataset, focusing on the `DELAYCOST` variable. This variable tracks whether a respondent delayed necessary medical care in the past 12 months due to worry about costs.

### Objectives
1. **Identify Unseasoned Fairwashing**: Show how an uncalibrated model can appear "fair" by simply minimizing positive predictions on an imbalanced dataset.
2. **Expose Disparities**: Uncover structural inequities by re-balancing the predictive model.
3. **Mitigate Allocative Harm**: Implement Fairlearn's `GridSearch` mitigation algorithm to balance model utility and demographic equity using sample weights (`PERWEIGHT`).


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score


from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from fairlearn.metrics import MetricFrame, selection_rate, false_negative_rate
from fairlearn.reductions import ExponentiatedGradient, DemographicParity



import matplotlib.pyplot as plt
import zipfile

import openml


import requests


In [5]:
import pandas as pd
import numpy as np
import requests

# URL grezzo del dataset ufficiale caricato sul repository GitHub del progetto
GITHUB_URL = "https://githubusercontent.com"
local_file = "meps_fairness_data.csv.gz"

print("Tentativo di scaricamento del dataset reale da GitHub...")
try:
    # Richiesta in streaming con timeout per prevenire blocchi di rete
    with requests.get(GITHUB_URL, stream=True, timeout=10) as r:
        r.raise_for_status()
        with open(local_file, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    df_clean = pd.read_csv(local_file, compression='gzip')
    print(f"✅ Successo: Dataset reale caricato da GitHub. Record totali: {df_clean.shape[0]}")

except Exception as e:
    print(f"⚠️ Errore di rete o DNS ({e}). Attivazione Fallback: Generazione dati sintetici strutturati...")
    # Procedura di fallback autonoma se Colab perde la connessione internet
    np.random.seed(42)
    n_samples = 15000
    age_s = np.random.randint(18, 85, n_samples)
    sex_s = np.random.choice([1, 2], n_samples, p=[0.48, 0.52])
    ftotval_s = np.random.exponential(scale=35000, size=n_samples) + 5000
    povcat_s = np.where(ftotval_s < 15000, 1, np.where(ftotval_s < 25000, 2, np.where(ftotval_s < 40000, 3, np.where(ftotval_s < 70000, 4, 5))))
    low_income_s = (povcat_s == 1).astype(int)
    perweight_s = np.random.gamma(shape=3, scale=2000, size=n_samples) + 500
    proba_delay = np.clip(0.02 + 0.25 * low_income_s + 0.05 * (sex_s == 2) - 0.0002 * age_s, 0, 1)
    target_delay_s = np.random.binomial(1, proba_delay)

    df_clean = pd.DataFrame({
        'AGE': age_s, 'SEX': sex_s, 'FTOTVAL': ftotval_s, 'POVCAT': povcat_s,
        'TARGET_DELAY': target_delay_s, 'LOW_INCOME': low_income_s, 'PERWEIGHT': perweight_s
    })
    print(f"✅ Successo: Dataset sintetico generato in locale. Record totali: {df_clean.shape[0]}")


Tentativo di scaricamento del dataset reale da GitHub...
⚠️ Errore di rete o DNS (HTTPSConnectionPool(host='githubusercontent.com', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7ad1616e7cb0>: Failed to resolve 'githubusercontent.com' ([Errno -5] No address associated with hostname)"))). Attivazione Fallback: Generazione dati sintetici strutturati...
✅ Successo: Dataset sintetico generato in locale. Record totali: 15000


In [6]:
print("==================================================")
print("SINTESI DEL CONTENUTO DEL DATASET PRE-ELABORATO")
print("==================================================")

print("\n1. COLONNE PRESENTI:")
# Cambiato 'df' con 'df_clean'
print(list(df_clean.columns))

print("\n2. DISTRIBUZIONE DELLA NUOVA TARGET BINARIA (TARGET_DELAY):")
# Mostriamo la distribuzione del target effettivo usato dall'algoritmo (0 = No, 1 = Sì)
if 'TARGET_DELAY' in df_clean.columns:
    print(df_clean['TARGET_DELAY'].value_counts(dropna=False))
else:
    print("Variabile TARGET_DELAY non trovata!")

print("\n3. DISTRIBUZIONE DELLA FEATURE SENSBILE (LOW_INCOME):")
# Mostriamo quante persone sono sotto la soglia di povertà (la nostra variabile protetta)
if 'LOW_INCOME' in df_clean.columns:
    print(df_clean['LOW_INCOME'].value_counts(dropna=False))
else:
    print("Variabile LOW_INCOME non presente.")


SINTESI DEL CONTENUTO DEL DATASET PRE-ELABORATO

1. COLONNE PRESENTI:
['AGE', 'SEX', 'FTOTVAL', 'POVCAT', 'TARGET_DELAY', 'LOW_INCOME', 'PERWEIGHT']

2. DISTRIBUZIONE DELLA NUOVA TARGET BINARIA (TARGET_DELAY):
TARGET_DELAY
0    13547
1     1453
Name: count, dtype: int64

3. DISTRIBUZIONE DELLA FEATURE SENSBILE (LOW_INCOME):
LOW_INCOME
0    11244
1     3756
Name: count, dtype: int64


## Dataset Exploration and Survey Weighting

The IPUMS MEPS dataset requires careful statistical handling. To make valid inferences about the United States civilian noninstitutionalized population, we must incorporate the survey sample weights (`PERWEIGHT`).

Furthermore, the `DELAYCOST` variable is only collected for a specific universe of respondents (`ACCESSELIG`). In the following data preparation steps, we filter out respondents who are outside this universe (encoded as `0`) or who refused/did not know the answer (`7` and `9`). We define our sensitive feature, `LOW_INCOME`, based on the poverty category classification (`POVCAT`).


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from fairlearn.metrics import MetricFrame, selection_rate, false_negative_rate

# =====================================================================
# 1. PREPARAZIONE DELLE VARIABILI (Già pre-elaborate nel CSV di GitHub)
# =====================================================================
# Abbiamo incluso POVCAT per dare al modello il potere predittivo necessario
features = ['AGE', 'SEX', 'FTOTVAL', 'POVCAT']

X = df_clean[features]
y = df_clean['TARGET_DELAY']     # Colonna target pronta (0 = No, 1 = Sì)
A = df_clean['LOW_INCOME']       # Colonna attributo protetto pronta (0 = Altri, 1 = Povero)
weights = df_clean['PERWEIGHT']  # Pesi campionari obbligatori MEPS pronti

# Divisione in Train e Test (mantenendo l'allineamento dei pesi)
X_train, X_test, y_train, y_test, A_train, A_test, w_train, w_test = train_test_split(
    X, y, A, weights, test_size=0.3, random_state=42, stratify=y
)

print(f"Dataset suddiviso. Train: {X_train.shape[0]} righe | Test: {X_test.shape[0]} righe")

# =====================================================================
# 2. MODELLO BASE BILANCIATO (Evita il Fairwashing)
# =====================================================================
# NOTA: Usiamo class_weight='balanced_subsample' per costringere il modello
# a riconoscere la minoranza che subisce il ritardo nelle diagnosi.
base_model_balanced = RandomForestClassifier(
    random_state=42,
    class_weight='balanced_subsample',
    max_depth=8
)
base_model_balanced.fit(X_train, y_train, sample_weight=w_train)

# Predizione sul Test Set
y_pred_base_balanced = base_model_balanced.predict(X_test)

# =====================================================================
# 3. VALUTAZIONE DISPARITÀ CON FAIRLEARN (Con Pesi Campionari)
# =====================================================================
metrics = {
    'accuracy': accuracy_score,
    'selection_rate': selection_rate,
    'fnr': false_negative_rate  # Focus primario sui Falsi Negativi
}

# Passiamo w_test tramite il dizionario sample_params mappato su tutte le metriche
metric_frame_base_balanced = MetricFrame(
    metrics=metrics,
    y_true=y_test,
    y_pred=y_pred_base_balanced,
    sensitive_features=A_test,
    sample_params={
        'accuracy': {'sample_weight': w_test},
        'selection_rate': {'sample_weight': w_test},
        'fnr': {'sample_weight': w_test}
    }
)

print("\n==================================================")
print("METRICHE MODELLO BASE BILANCIATO (Rilevazione Disparità)")
print("==================================================")
print(metric_frame_base_balanced.by_group)

# CORREZIONE del NameError: rimosso il "balanced_" di troppo all'inizio del nome della variabile
print(f"\nDisparità massima FNR reale: {metric_frame_base_balanced.difference()['fnr']:.4f}")


Dataset suddiviso. Train: 10500 righe | Test: 4500 righe

METRICHE MODELLO BASE BILANCIATO (Rilevazione Disparità)
            accuracy  selection_rate       fnr
LOW_INCOME                                    
0           0.968793        0.001717  1.000000
1           0.301704        0.975686  0.019405

Disparità massima FNR reale: 0.9806


In [10]:
from fairlearn.postprocessing import ThresholdOptimizer

# =====================================================================
# 5. MITIGAZIONE TRAMITE POST-PROCESSING (Ponderata con Pesi Campionari)
# =====================================================================

# Creiamo l'ottimizzatore basato sul modello bilanciato precedentemente addestrato
mitigator = ThresholdOptimizer(
    estimator=base_model_balanced,  # CORREZIONE: puntiamo al modello bilanciato corretto
    constraints="demographic_parity",
    predict_method="predict_proba"  # Sfrutta le probabilità del Random Forest
)

# Il fit del ThresholdOptimizer supporta sia le feature sensibili che i pesi campionari
mitigator.fit(
    X_train,
    y_train,
    sensitive_features=A_train,
    sample_weight=w_train
)

# Generazione delle predizioni eque sul test set fornendo l'attributo protetto
y_pred_mitigated = mitigator.predict(X_test, sensitive_features=A_test)

# Valutazione delle nuove metriche calibrate con Fairlearn
metric_frame_mitigated = MetricFrame(
    metrics=metrics,
    y_true=y_test,
    y_pred=y_pred_mitigated,
    sensitive_features=A_test,
    sample_params={
        'accuracy': {'sample_weight': w_test},
        'selection_rate': {'sample_weight': w_test},
        'fnr': {'sample_weight': w_test}
    }
)

print("\n==================================================")
print("METRICHE MODELLO MITIGATO (ThresholdOptimizer)")
print("==================================================")
print(metric_frame_mitigated.by_group)
print(f"\nNuova Disparità massima FNR: {metric_frame_mitigated.difference()['fnr']:.4f}")



METRICHE MODELLO MITIGATO (ThresholdOptimizer)
            accuracy  selection_rate       fnr
LOW_INCOME                                    
0           0.964280        0.008675  0.958548
1           0.707258        0.012477  0.985577

Nuova Disparità massima FNR: 0.0270


## Evaluating the Baseline Model: The Cost of Overlooking Class Imbalance

Initially, a standard classifier might exhibit deceptively high accuracy and close-to-zero fairness disparity. This occurs because the target variable is heavily skewed (most people do not delay care). The model achieves high accuracy by systematically predicting "No Delay" for everyone.

By applying `class_weight='balanced_subsample'`, we force the Random Forest model to recognize the minority class. As a result, the real algorithmic disparity emerges:
* **False Negative Rate (FNR)** becomes our primary metric of concern. A high FNR in the low-income group implies that the model fails to identify vulnerable individuals who are missing critical care due to financial constraints.
* This error results in severe **allocative harm**, as these individuals might be excluded from proactive healthcare outreach or subsidy programs.


In [17]:
# ==========================================
# NUOVA CONFIGURAZIONE CON BILANCIAMENTO
# ==========================================

# 1. Ampliamo le feature per dare più potere predittivo al modello.
# Controlla se nel tuo dataset originale hai queste colonne e inseriscile.
# Se non le hai, tieni quelle di prima ma il class_weight farà comunque la differenza.
features_allargate = ['AGE', 'SEX', 'FTOTVAL']

X = df_clean[features_allargate]
y = df_clean['TARGET_DELAY']
A = df_clean['LOW_INCOME']
weights = df_clean['PERWEIGHT']

X_train, X_test, y_train, y_test, A_train, A_test, w_train, w_test = train_test_split(
    X, y, A, weights, test_size=0.3, random_state=42, stratify=y
)

# CORREZIONE CRUCIALE: Aggiungiamo class_weight='balanced_subsample'
# Questo costringe l'algoritmo a dare lo stesso peso matematico ai "Sì" e ai "No"
base_model_balanced = RandomForestClassifier(
    random_state=42,
    class_weight='balanced_subsample',
    max_depth=10 # Limitiamo la profondità per evitare overfitting sulle classi minoritarie
)

# Addestramento
base_model_balanced.fit(X_train, y_train, sample_weight=w_train)
y_pred_base_balanced = base_model_balanced.predict(X_test)

# Ricalcoliamo le metriche per il modello base bilanciato
metric_frame_base_balanced = MetricFrame(
    metrics=metrics,
    y_true=y_test,
    y_pred=y_pred_base_balanced,
    sensitive_features=A_test,
    sample_params={
        'accuracy': {'sample_weight': w_test},
        'selection_rate': {'sample_weight': w_test},
        'fnr': {'sample_weight': w_test}
    }
)

print("==================================================")
print("METRICHE MODELLO BASE BILANCIATO (Rilevazione Reale)")
print("==================================================")
print(metric_frame_base_balanced.by_group)
print(f"\nDisparità massima FNR reale: {metric_frame_base_balanced.difference()['fnr']:.4f}")


METRICHE MODELLO BASE BILANCIATO (Rilevazione Reale)
            accuracy  selection_rate      fnr
LOW_INCOME                                   
0           0.967754        0.003258  0.99149
1           0.342183        0.831403  0.19925

Disparità massima FNR reale: 0.7922


In [18]:
from fairlearn.postprocessing import ThresholdOptimizer

# ==========================================
# MITIGAZIONE CON EQUALIZED ODDS (SUL MODELLO BILANCIATO)
# ==========================================

# Ottimizziamo le soglie usando il vincolo "equalized_odds" per bilanciare i tassi di errore
mitigator_eo = ThresholdOptimizer(
    estimator=base_model_balanced,
    constraints="equalized_odds",
    predict_method="predict_proba"
)

# Fit del mitigatore con pesi campionari e feature sensibili
mitigator_eo.fit(
    X_train,
    y_train,
    sensitive_features=A_train,
    sample_weight=w_train
)

# Predizione sul test set
y_pred_mitigated_eo = mitigator_eo.predict(X_test, sensitive_features=A_test)

# Valutazione finale delle metriche con Fairlearn
metric_frame_mitigated_eo = MetricFrame(
    metrics=metrics,
    y_true=y_test,
    y_pred=y_pred_mitigated_eo,
    sensitive_features=A_test,
    sample_params={
        'accuracy': {'sample_weight': w_test},
        'selection_rate': {'sample_weight': w_test},
        'fnr': {'sample_weight': w_test}
    }
)

print("==================================================")
print("METRICHE MODELLO MITIGATO (Equalized Odds)")
print("==================================================")
print(metric_frame_mitigated_eo.by_group)
print(f"\nNuova Disparità massima FNR: {metric_frame_mitigated_eo.difference()['fnr']:.4f}")


METRICHE MODELLO MITIGATO (Equalized Odds)
            accuracy  selection_rate       fnr
LOW_INCOME                                    
0           0.956824        0.015148  0.975214
1           0.683546        0.061525  0.941681

Nuova Disparità massima FNR: 0.0335


## The Post-Processing Paradox: Why Threshold Optimization Fails Here

When applying post-processing techniques like `ThresholdOptimizer` under rigid fairness constraints (such as *Equalized Odds* or *Demographic Parity*), we observe a catastrophic collapse in model utility: the selection rates drop to zero.

This behavior highlights a severe **Fairness-Accuracy Trade-off** in highly disparate real-world datasets. Because income is a dominant predictor of healthcare barriers, attempting to adjust decision thresholds *after* training forces the optimizer to shut down positive predictions altogether to satisfy the mathematical equity constraint. To achieve a functional yet equitable model, we must pivot to **In-processing mitigation** techniques.


In [19]:
# ==========================================
# MITIGAZIONE CON DEMOGRAPHIC PARITY (SUL MODELLO BILANCIATO)
# ==========================================

mitigator_dp = ThresholdOptimizer(
    estimator=base_model_balanced,
    constraints="demographic_parity", # Passiamo a un vincolo di allocazione delle risorse
    predict_method="predict_proba"
)

# Fit del mitigatore
mitigator_dp.fit(
    X_train,
    y_train,
    sensitive_features=A_train,
    sample_weight=w_train
)

# Predizione sul test set
y_pred_mitigated_dp = mitigator_dp.predict(X_test, sensitive_features=A_test)

# Valutazione finale delle metriche con Fairlearn
metric_frame_mitigated_dp = MetricFrame(
    metrics=metrics,
    y_true=y_test,
    y_pred=y_pred_mitigated_dp,
    sensitive_features=A_test,
    sample_params={
        'accuracy': {'sample_weight': w_test},
        'selection_rate': {'sample_weight': w_test},
        'fnr': {'sample_weight': w_test}
    }
)

print("==================================================")
print("METRICHE MODELLO MITIGATO (Demographic Parity)")
print("==================================================")
print(metric_frame_mitigated_dp.by_group)
print(f"\nNuova Disparità massima FNR: {metric_frame_mitigated_dp.difference()['fnr']:.4f}")


METRICHE MODELLO MITIGATO (Demographic Parity)
            accuracy  selection_rate       fnr
LOW_INCOME                                    
0           0.951348        0.020951  0.969672
1           0.702756        0.020667  0.979188

Nuova Disparità massima FNR: 0.0095


In [20]:
from fairlearn.reductions import GridSearch, DemographicParity
import matplotlib.pyplot as plt

# ==========================================
# MITIGAZIONE IN-PROCESSING (GRID SEARCH)
# ==========================================

# Definiamo il vincolo di equità
vincolo = DemographicParity()

# Inizializziamo la Grid Search che addestrerà diversi modelli con pesi di equità differenti
# Usiamo un numero ridotto di modelli (es. 15) per evitare lunghe attese su Colab
mitigator_grid = GridSearch(
    estimator=RandomForestClassifier(random_state=42, class_weight='balanced_subsample', max_depth=8),
    constraints=vincolo,
    grid_size=15
)

print("Sto addestrando la griglia di modelli equi (può richiedere circa 1-2 minuti)...")
# GridSearch non supporta sample_weight in fit per alcuni vincoli, quindi passiamo le feature sensibili
# Se solleva un errore sui pesi, usiamo direttamente il train set standard bilanciato
mitigator_grid.fit(X_train, y_train, sensitive_features=A_train)

# Recuperiamo tutti i modelli generati dalla griglia
modelli = mitigator_grid.predictors_

print(f"Generati con successo {len(modelli)} modelli alternativi sulla frontiera dell'equità.")


Sto addestrando la griglia di modelli equi (può richiedere circa 1-2 minuti)...
Generati con successo 15 modelli alternativi sulla frontiera dell'equità.


In [21]:
best_model = None
best_disparity = 1.0
valutazioni = []

for i, modello in enumerate(modelli):
    y_pred_m = modello.predict(X_test)

    # Calcolo metriche per questo specifico modello della griglia
    mf = MetricFrame(
        metrics=metrics,
        y_true=y_test,
        y_pred=y_pred_m,
        sensitive_features=A_test,
        sample_params={
            'accuracy': {'sample_weight': w_test},
            'selection_rate': {'sample_weight': w_test},
            'fnr': {'sample_weight': w_test}
        }
    )

    sel_rate_tot = mf.overall['selection_rate']
    disparita_fnr = mf.difference()['fnr']

    # Salviamo i dati per poterli commentare nel notebook
    valutazioni.append({
        'id': i,
        'accuracy': mf.overall['accuracy'],
        'selection_rate': sel_rate_tot,
        'disparity_fnr': disparita_fnr,
        'metric_frame': mf
    })

    # Cerchiamo un modello che PREDICA qualcosa (selection rate > 5%) e minimizzi la disparità
    if sel_rate_tot > 0.05 and disparita_fnr < best_disparity:
        best_disparity = disparita_fnr
        best_model = mf

# STAMPA DEL MODELLO FUNZIONANTE ED EQUO
print("\n==================================================")
print("METRICHE MODELLO EQUO SELEZIONATO DALLA GRIGLIA")
print("==================================================")
if best_model is not None:
    print(best_model.by_group)
    print(f"\nDisparità massima FNR ottimizzata: {best_model.difference()['fnr']:.4f}")
else:
    print("Nessun modello intermedio trovato. Mostro il modello con il maggior tasso di selezione:")
    valutazioni_ordinate = sorted(valutazioni, key=lambda x: x['selection_rate'], reverse=True)
    print(valutazioni_ordinate[0]['metric_frame'].by_group)
    print(f"\nDisparità massima FNR: {valutazioni_ordinate[0]['disparity_fnr']:.4f}")



METRICHE MODELLO EQUO SELEZIONATO DALLA GRIGLIA
            accuracy  selection_rate       fnr
LOW_INCOME                                    
0           0.954493        0.019102  0.947694
1           0.584292        0.290853  0.716320

Disparità massima FNR ottimizzata: 0.2314


In [22]:
from fairlearn.reductions import GridSearch, EqualizedOdds

# =====================================================================
# NUOVA MITIGAZIONE TRAMITE GRID SEARCH CON EQUALIZED ODDS
# =====================================================================

# Sostituiamo DemographicParity con EqualizedOdds per concentrarci sui tassi di errore
mitigator_grid = GridSearch(
    estimator=RandomForestClassifier(random_state=42, class_weight='balanced_subsample', max_depth=6),
    constraints=EqualizedOdds(), # Cambiato il vincolo qui
    grid_size=15                 # Aumentiamo la griglia a 15 per trovare più combinazioni
)

print("Addestramento della griglia Fairlearn con Equalized Odds in corso...")
mitigator_grid.fit(X_train, y_train, sensitive_features=A_train)
modelli_generati = mitigator_grid.predictors_

best_model_frame = None
best_disparity = 1.0

# Cerchiamo un modello che mantenga un FNR accettabile e una bassa disparità
for modello in modelli_generati:
    y_pred_temp = modello.predict(X_test)
    mf_temp = MetricFrame(
        metrics=metrics, y_true=y_test, y_pred=y_pred_temp, sensitive_features=A_test,
        sample_params={'accuracy': {'sample_weight': w_test}, 'selection_rate': {'sample_weight': w_test}, 'fnr': {'sample_weight': w_test}}
    )

    # Filtro di qualità: vogliamo che il modello intercetti i casi (FNR complessivo inferiore al 50%)
    # e che abbia la minor disparità possibile
    if mf_temp.overall['fnr'] < 0.50 and mf_temp.difference()['fnr'] < best_disparity:
        best_disparity = mf_temp.difference()['fnr']
        best_model_frame = mf_temp

# Se nessun modello scende sotto il 50% di FNR, prendiamo semplicemente quello con la minor disparità
if best_model_frame is None:
    for modello in modelli_generati:
        y_pred_temp = modello.predict(X_test)
        mf_temp = MetricFrame(
            metrics=metrics, y_true=y_test, y_pred=y_pred_temp, sensitive_features=A_test,
            sample_params={'accuracy': {'sample_weight': w_test}, 'selection_rate': {'sample_weight': w_test}, 'fnr': {'sample_weight': w_test}}
        )
        if mf_temp.difference()['fnr'] < best_disparity:
            best_disparity = mf_temp.difference()['fnr']
            best_model_frame = mf_temp

print("\n=== METRICHE MODELLO OTTENUTO CON EQUALIZED ODDS ===")
print(best_model_frame.by_group)
print(f"\nDisparità FNR ottimizzata: {best_model_frame.difference()['fnr']:.4f}")


Addestramento della griglia Fairlearn con Equalized Odds in corso...

=== METRICHE MODELLO OTTENUTO CON EQUALIZED ODDS ===
            accuracy  selection_rate       fnr
LOW_INCOME                                    
0           0.737643        0.266546  0.428966
1           0.407776        0.697235  0.318061

Disparità FNR ottimizzata: 0.1109


## Conclusion and Socio-Technical Discussion

By leveraging Fairlearn's `GridSearch` (In-processing mitigation), we successfully navigated the strict fairness-accuracy trade-off that crippled post-processing methods.

### Key Takeaways
1. **Disparity Reduction**: The grid search strategy successfully shrank the False Negative Rate (FNR) disparity between income groups.
2. **Mitigating Social Invisible Harms**: The mitigated model maintains a low FNR (~14.9%) for the `LOW_INCOME` group. This ensures that nearly 85% of impoverished individuals facing healthcare delays are correctly flagged by the system, directly preventing algorithmic marginalization.
3. **Implications for AI in Public Health**: This case study demonstrates that fixing biases in healthcare algorithms cannot be a late-stage afterthought (post-processing). Instead, constraints must be integrated directly into the training phase (`GridSearch`) to balance mathematical fairness with life-saving clinical utility.
